# Lab 2 — Stock forecasting with RNN, LSTM and GRU
AAPL daily OHLCV; chronological evaluation. Educational exercise, not financial advice.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','-q','install','yfinance','mplfinance','statsmodels'])

In [ ]:
import os, random, copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yfinance as yf, torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import DataLoader, TensorDataset
from statsmodels.graphics.tsaplots import plot_acf
import mplfinance as mpf
torch.set_num_threads(min(2,os.cpu_count() or 2))
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Torch',torch.__version__,'Device',DEVICE)

## Download and cache data
Daily OHLCV, minimum five years. Yahoo Finance data can change on subsequent downloads.

In [ ]:
os.makedirs('data',exist_ok=True); os.makedirs('checkpoints',exist_ok=True)
p='data/AAPL_ohlcv.csv'
if os.path.exists(p): df=pd.read_csv(p,index_col=0,parse_dates=True)
else:
    df=yf.download('AAPL',period='7y',auto_adjust=True,progress=False)
    if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(0)
    df=df[['Open','High','Low','Close','Volume']].dropna(); df.to_csv(p)
df.index=pd.to_datetime(df.index,utc=True).tz_localize(None)
df=df.sort_index().dropna()
assert len(df)>252*5, 'Insufficient daily history'
print(df.shape,df.index.min(),df.index.max()); display(df.tail())

## Feature engineering and chronological splits
Rolling features use past/current data only. Scalers are fitted on training observations exclusively.

In [ ]:
df['MA20']=df.Close.rolling(20).mean(); df['MA50']=df.Close.rolling(50).mean()
df['VOL20']=df.Close.pct_change().rolling(20).std()
df['Range']=(df.High-df.Low)/df.Close
df=df.dropna().copy()
n=len(df); a=int(n*.7); b=int(n*.85); LOOKBACK=60
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(df.index,df.Close); ax.axvspan(df.index[0],df.index[a-1],alpha=.12,label='train'); ax.axvspan(df.index[a],df.index[b-1],alpha=.12,label='validation'); ax.axvspan(df.index[b],df.index[-1],alpha=.12,label='test'); ax.legend(); ax.set_title('AAPL adjusted Close — chronological splits'); plt.show()
mpf.plot(df[['Open','High','Low','Close','Volume']].tail(126),type='candle',volume=True,style='yahoo',title='AAPL recent candles')
df[['Close','MA20','MA50']].plot(figsize=(12,4),title='Close and moving averages'); plt.show()

## Sliding windows and baselines
Targets correspond to the next trading day. Test windows may use earlier observed context, never future targets.

In [ ]:
def windows(cols):
    sx=StandardScaler().fit(df.iloc[:a][cols]); sy=StandardScaler().fit(df.iloc[:a][['Close']])
    X=sx.transform(df[cols]).astype('float32'); y=sy.transform(df[['Close']]).astype('float32').ravel()
    xx=[]; yy=[]; ids=[]
    for t in range(LOOKBACK,len(df)):
        xx.append(X[t-LOOKBACK:t]); yy.append(y[t]); ids.append(t)
    return np.array(xx),np.array(yy),np.array(ids),sy,sx

def baselines(ids):
    actual=df.Close.to_numpy(); last=actual[ids-1]; ma=np.array([actual[t-20:t].mean() for t in ids]); return last,ma

def metrics(y,p):
    return {'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p))),'MAPE%':float(np.mean(np.abs((y-p)/np.maximum(np.abs(y),1e-8)))*100)}
print('Last value baseline',metrics(df.Close.to_numpy()[np.arange(b,n)],baselines(np.arange(b,n))[0]))

## PyTorch recurrent architectures
Each architecture uses the same input windows and prediction target.

In [ ]:
class ForecastNet(nn.Module):
    def __init__(self,kind,n_features,hidden=32):
        super().__init__(); self.recurrent={'RNN':nn.RNN,'LSTM':nn.LSTM,'GRU':nn.GRU}[kind](n_features,hidden,batch_first=True); self.head=nn.Linear(hidden,1)
    def forward(self,x):
        out,_=self.recurrent(x); return self.head(out[:,-1,:]).squeeze(-1)

def train(kind,cols,epochs=35,patience=8):
    X,y,ids,sy,sx=windows(cols)
    tr=ids<a; va=(ids>=a)&(ids<b); te=ids>=b
    loader=DataLoader(TensorDataset(torch.tensor(X[tr]),torch.tensor(y[tr])),batch_size=64,shuffle=True)
    net=ForecastNet(kind,len(cols)).to(DEVICE); opt=torch.optim.Adam(net.parameters(),lr=.001)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=3)
    lossfn=nn.MSELoss(); best=float('inf'); wait=0; history=[]; best_state=None
    for epoch in range(epochs):
        net.train(); total=0
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=lossfn(net(xb),yb); loss.backward(); opt.step(); total+=loss.item()*len(xb)
        net.eval()
        with torch.no_grad():
            vl=lossfn(net(torch.tensor(X[va],device=DEVICE)),torch.tensor(y[va],device=DEVICE)).item()
        scheduler.step(vl)
        with torch.no_grad(): val_scaled=net(torch.tensor(X[va],device=DEVICE)).cpu().numpy()
        val_pred=sy.inverse_transform(val_scaled.reshape(-1,1)).ravel()
        val_rmse=float(np.sqrt(np.mean((val_pred-df.Close.to_numpy()[ids[va]])**2)))
        history.append((total/tr.sum(),vl,val_rmse))
        if vl<best-1e-6: best=vl; wait=0; best_state=copy.deepcopy(net.state_dict())
        else:
            wait+=1
            if wait>=patience: break
    net.load_state_dict(best_state)
    name=f'{kind}_{len(cols)}features_L{LOOKBACK}'; torch.save({'state_dict':best_state,'features':cols,'kind':kind,'lookback':LOOKBACK,'scaler_x_mean':sx.mean_.tolist(),'scaler_x_scale':sx.scale_.tolist(),'scaler_y_mean':sy.mean_.tolist(),'scaler_y_scale':sy.scale_.tolist()},f'checkpoints/{name}.pt')
    net.eval()
    with torch.no_grad(): pred=net(torch.tensor(X[te],device=DEVICE)).cpu().numpy()
    prediction=sy.inverse_transform(pred.reshape(-1,1)).ravel(); actual=df.Close.to_numpy()[ids[te]]
    with torch.no_grad(): vp=net(torch.tensor(X[va],device=DEVICE)).cpu().numpy()
    validation_pred=sy.inverse_transform(vp.reshape(-1,1)).ravel()
    validation_actual=df.Close.to_numpy()[ids[va]]
    return name,metrics(actual,prediction),history,ids[te],actual,prediction,metrics(validation_actual,validation_pred)

## Experiments
RNN, LSTM, GRU on univariate and multivariate features. Report test metrics only after training/selection on validation.

## Hyperparameter ablation: lookback window
Two additional univariate LSTM fits use 30-day and 90-day input windows (versus the main 60-day setting). Their model selection uses the same validation period. Training settings other than lookback remain unchanged; the original 60-day setting is restored before forecast reload and walk-forward evaluation.

In [ ]:
experiments=[]
for cols in [['Close'],['Close','Volume','MA20','MA50','VOL20','Range']]:
    for kind in ['RNN','LSTM','GRU']:
        result=train(kind,cols); experiments.append(result); print(result[0],result[1])
for alternate_lookback in [30,90]:
    LOOKBACK=alternate_lookback
    result=train('LSTM',['Close'])
    experiments.append(result)
    print('Lookback ablation',alternate_lookback,result[6])
LOOKBACK=60
results=pd.DataFrame([{'model':x[0],**{'test_'+k:v for k,v in x[1].items()},**{'validation_'+k:v for k,v in x[6].items()}} for x in experiments]); display(results.sort_values('validation_RMSE'))
ids=experiments[0][3]; actual=experiments[0][4]; last,ma=baselines(ids)
baseline_results=pd.DataFrame([{'model':'Last value',**metrics(actual,last)},{'model':'Moving average 20',**metrics(actual,ma)}]); display(baseline_results)
assert len(ids)==len(actual)==len(last)==len(ma)
os.makedirs('reports',exist_ok=True)
results.to_csv('reports/model_metrics.csv',index=False)
baseline_results.to_csv('reports/baseline_metrics.csv',index=False)

## Training curves, prediction comparison and residual diagnostics

In [ ]:
best=min(experiments,key=lambda r:r[6]['RMSE']); name,score,history,ids,actual,pred=best
fig,axs=plt.subplots(2,4,figsize=(18,7))
for ax,exp in zip(axs.ravel(),experiments):
    h=np.array(exp[2]); ax.plot(h[:,0],label='train loss'); ax.plot(h[:,1],label='validation loss'); ax.set_title(exp[0]); ax.set_xlabel('Epoch'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
fig,axs=plt.subplots(2,4,figsize=(18,7))
for ax,exp in zip(axs.ravel(),experiments):
    h=np.array(exp[2]); ax.plot(h[:,2],label='validation RMSE (USD)'); ax.set_title(exp[0]); ax.set_xlabel('Epoch'); ax.set_ylabel('USD'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(df.index[ids],actual,label='actual'); ax.plot(df.index[ids],pred,label='prediction'); ax.set_title('Test: '+name); ax.legend(); plt.show()
res=actual-pred
spread=1.96*np.std(res,ddof=1)
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(df.index[ids],actual,label='actual'); ax.plot(df.index[ids],pred,label='prediction'); ax.fill_between(df.index[ids],pred-spread,pred+spread,alpha=.16,label='±1.96 residual SD, descriptive (not calibrated)'); ax.legend(); ax.set_title('Test predictions with empirical residual spread'); plt.show()
pd.DataFrame({'Date':df.index[ids],'Actual':actual,'Prediction':pred,'Residual':res}).to_csv('reports/test_predictions.csv',index=False)
fig,ax=plt.subplots(figsize=(10,3)); ax.plot(df.index[ids],res); ax.set_title('Residual time series'); plt.show()
plt.hist(res,bins=30); plt.title('Residual histogram'); plt.show()
plot_acf(res,lags=min(30,len(res)//3)); plt.show()

## Walk-forward validation schematic
This illustrates expanding training and rolling validation windows; a full refit-and-score walk-forward evaluation remains to be implemented.

In [ ]:
fig,ax=plt.subplots(figsize=(10,2.5))
for i in range(4):
    ax.broken_barh([(0,55+10*i)],(i-.3,.25),facecolors='steelblue'); ax.broken_barh([(55+10*i,10)],(i-.3,.25),facecolors='orange')
ax.set_yticks(range(4),[f'Fold {i+1}' for i in range(4)]); ax.set_xlabel('Illustrative chronological index'); ax.set_title('Walk-forward scheme (diagram only)'); plt.show()

## Reload model and next-step forecast
Uses the last observed window. A prediction is not a trading recommendation.

In [ ]:
def reload_and_forecast(checkpoint_path,history_df):
    obj=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    features=obj['features']; model=ForecastNet(obj['kind'],len(features)); model.load_state_dict(obj['state_dict']); model.eval()
    x=history_df[features].tail(obj['lookback']).to_numpy(dtype='float32')
    if len(x)!=obj['lookback']: raise ValueError('Insufficient history')
    x=((x-np.array(obj['scaler_x_mean'],dtype='float32'))/np.array(obj['scaler_x_scale'],dtype='float32')).astype('float32')
    with torch.no_grad(): scaled=model(torch.tensor(x[None,:,:],dtype=torch.float32)).item()
    return scaled*obj['scaler_y_scale'][0]+obj['scaler_y_mean'][0]
selected_forecast=reload_and_forecast('checkpoints/'+name+'.pt',df)
best_60=min((r for r in experiments if r[0].endswith('_L60')),key=lambda r:r[6]['RMSE'])
forecast_60=reload_and_forecast('checkpoints/'+best_60[0]+'.pt',df.tail(60))
assert np.isfinite(selected_forecast) and np.isfinite(forecast_60)
print('Selected-model next trading day estimate:',selected_forecast)
print('Required 60-day reload demonstration:',best_60[0],forecast_60)

## Expanding-window walk-forward evaluation
Repeatedly refit a univariate LSTM using only historical training data at each fold. Evaluate on the following chronological validation segment and report aggregate fold errors. The final held-out test period is never used in this procedure.

In [ ]:
def walk_forward_lstm(folds=3,epochs=12):
    all_metrics=[]
    edges=np.linspace(int(len(df)*.55),b,folds+1,dtype=int)
    for k in range(folds):
        train_end=int(edges[k]); valid_end=int(edges[k+1])
        if valid_end<=train_end: continue
        close=df[['Close']].to_numpy(dtype='float32')
        scaler=StandardScaler().fit(close[:train_end])
        scaled=scaler.transform(close).ravel()
        xx=np.stack([scaled[t-LOOKBACK:t] for t in range(LOOKBACK,valid_end)]).astype('float32')
        yy=scaled[LOOKBACK:valid_end].astype('float32')
        ids=np.arange(LOOKBACK,valid_end)
        tr=ids<train_end; va=(ids>=train_end)&(ids<valid_end)
        model=ForecastNet('LSTM',1).to(DEVICE)
        optim=torch.optim.Adam(model.parameters(),lr=.001)
        train_loader=DataLoader(TensorDataset(torch.tensor(xx[tr]),torch.tensor(yy[tr])),batch_size=64,shuffle=True)
        for _ in range(epochs):
            model.train()
            for xbatch,ybatch in train_loader:
                xbatch,ybatch=xbatch.to(DEVICE),ybatch.to(DEVICE)
                optim.zero_grad(); loss=nn.functional.mse_loss(model(xbatch),ybatch); loss.backward(); optim.step()
        model.eval()
        with torch.no_grad():
            scaled_pred=model(torch.tensor(xx[va],device=DEVICE)).cpu().numpy()
        pred=scaler.inverse_transform(scaled_pred.reshape(-1,1)).ravel()
        truth=close[ids[va],0]
        row={'fold':k+1,'train_end':str(df.index[train_end-1].date()),'validation_end':str(df.index[valid_end-1].date()),**metrics(truth,pred)}
        all_metrics.append(row)
    return pd.DataFrame(all_metrics)
walk_forward_results=walk_forward_lstm()
display(walk_forward_results)
display(walk_forward_results[['MAE','RMSE','MAPE%']].agg(['mean','std']))

## Executive summary and conclusions
Complete after execution using measured test performance. Identify the best model, compare both baselines, explain the chronological validation strategy, and discuss nonstationarity, residual patterns and limitations.

## Reproducibility
Random seed: 42. Dataset is cached to CSV. Save outputs and checkpoints as workflow artifacts; do not interpret these forecasts as financial advice.

## Reproducible results and executive summary
The selected recurrent architecture is chosen **only by validation RMSE**. The held-out test metrics are reported for assessment, not used for model selection. The baseline comparison provides a reality check. The walk-forward folds are distinct from the final test period.

In [ ]:
from IPython.display import Markdown, display
baseline_best=baseline_results.sort_values('RMSE').iloc[0]
wfmean=walk_forward_results[['MAE','RMSE','MAPE%']].mean(numeric_only=True)
comparison='outperformed' if score['RMSE']<baseline_best['RMSE'] else 'did not outperform'
summary=f'''# Executive summary — AAPL next-day forecasting

**Research question.** Can a recurrent neural network predict the next day's Apple closing price more accurately than simple historical baselines?

**Data and method.** Yahoo Finance daily OHLCV data covering {df.index.min().date()} to {df.index.max().date()} ({len(df)} processed trading days). A strictly chronological 70%/15%/15% train/validation/test split, 60-day lookback, 1-day horizon, and training-only scaling were used. Three recurrent architectures (RNN, LSTM, GRU) were evaluated using univariate and multivariate feature sets. Checkpoints and early stopping are based on validation loss; final architecture selection is based on validation RMSE.

**Selected model.** {name}, validation RMSE **{best[6]['RMSE']:.3f} USD**. Its untouched final test scores were MAE **{score['MAE']:.3f} USD**, RMSE **{score['RMSE']:.3f} USD**, and MAPE **{score['MAPE%']:.2f}%**.

**Baselines.** The strongest naïve/moving-average baseline on test was {baseline_best['model']} with RMSE **{baseline_best['RMSE']:.3f} USD**. The selected recurrent model **{comparison}** this baseline on the test segment. A recurrent model should not be presumed superior to a last-value predictor.

**60-day reload demonstration.** {best_60[0]} successfully reloaded using only the last 60 observed trading days to estimate the following closing price (**{forecast_60:.2f} USD**); the exact forecasting date depends on the latest available trading session. This is not a trade recommendation.\n\n**Temporal robustness.** Expanding-window walk-forward LSTM refits yielded mean validation RMSE **{wfmean['RMSE']:.3f} USD** across {len(walk_forward_results)} folds. These are validation folds only and are not estimates from the final test interval.

**What worked and limitations.** Sliding windows, recurrent training, checkpoint reload, and comparisons are implemented. Financial prices are nonstationary: regime shifts, volatility and persistent residual structure may erode generalization. Residual spread is descriptive, not a calibrated prediction interval. Hyperparameter search is limited. Past prices and indicators do not establish causal forecastability.

**Reproducibility.** Seed 42; the workflow caches the original OHLCV CSV, exports metrics and prediction CSVs, archives PyTorch weights, and preserves the executed notebook. Results apply to the downloaded data snapshot and package versions captured by the runner. Not financial advice.
'''
with open('reports/executive_summary.md','w') as out: out.write(summary)
display(Markdown(summary))
import platform
with open('reports/environment.txt','w') as out: out.write(f'Python: {platform.python_version()}\nPyTorch: {torch.__version__}\nCUDA: {torch.version.cuda}\ncuDNN: {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else "not available"}\nDevice: {DEVICE}\nNumPy: {np.__version__}\nPandas: {pd.__version__}\n')